# Phase 1: Step 1.1 — NOAA IBTrACS Ground Truth Pipeline
### Project: DeepCyclone / CycloneAI

This notebook directly reads your locally downloaded `IBTrACS.NI.list.v04r00.csv` file, cleans the data, maps official IMD categories, flags Rapid Intensification, and splits the dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Local file path (put the CSV in the same folder, or specify its local path)
csv_path = "IBTrACS.NI.list.v04r00.csv"

# Row 0 = column names, Row 1 = units -> skiprows=[1]
df = pd.read_csv(csv_path, skiprows=[1], low_memory=False)

print("Loaded successfully!")
print("Shape:", df.shape)
df.head()

## 1. Select Key Columns & Filter for 1990–2024

In [ ]:
keep_columns = [
    'SID',          # Storm ID
    'SEASON',       # Year
    'NAME',         # Storm Name
    'ISO_TIME',     # Timestamp
    'LAT',          # Latitude
    'LON',          # Longitude
    'WMO_WIND',     # Sustained Wind (knots)
    'WMO_PRES',     # Central Pressure (hPa)
    'STORM_SPEED',  # Translation Speed (km/h)
    'STORM_DIR'     # Heading Direction (degrees)
]

df = df[keep_columns].copy()

# Convert types
df['SEASON'] = pd.to_numeric(df['SEASON'], errors='coerce')
df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')
df['LON'] = pd.to_numeric(df['LON'], errors='coerce')
df['WMO_WIND'] = pd.to_numeric(df['WMO_WIND'], errors='coerce')
df['WMO_PRES'] = pd.to_numeric(df['WMO_PRES'], errors='coerce')

# Filter 1990 to 2024 and remove missing values
df = df[(df['SEASON'] >= 1990) & (df['SEASON'] <= 2024)]
df = df.dropna(subset=['LAT', 'LON', 'WMO_WIND']).reset_index(drop=True)

print("Cleaned observations:", len(df))
print("Unique storms:", df['SID'].nunique())
df.head()

## 2. Map IMD Cyclone Categories

In [ ]:
def map_imd_category(wind):
    if wind < 17:
        return 'Low Pressure Area'
    elif wind <= 27:
        return 'Depression'
    elif wind <= 33:
        return 'Deep Depression'
    elif wind <= 47:
        return 'Cyclonic Storm'
    elif wind <= 63:
        return 'Severe Cyclonic Storm'
    elif wind <= 89:
        return 'Very Severe Cyclonic Storm'
    elif wind <= 119:
        return 'Extremely Severe Cyclonic Storm'
    else:
        return 'Super Cyclonic Storm'

df['IMD_CATEGORY'] = df['WMO_WIND'].apply(map_imd_category)
df['IMD_CATEGORY'].value_counts()

## 3. Flag Rapid Intensification (RI)

In [ ]:
# 4 steps ahead = 24 hours (for 6-hourly fixes)
df = df.sort_values(by=['SID', 'ISO_TIME']).reset_index(drop=True)

df['WIND_24H_AHEAD'] = df.groupby('SID')['WMO_WIND'].shift(-4)
df['WIND_CHANGE_24H'] = df['WIND_24H_AHEAD'] - df['WMO_WIND']
df['RI_EVENT'] = (df['WIND_CHANGE_24H'] >= 30).astype(int)

print("Total RI events:", df['RI_EVENT'].sum())
df[df['RI_EVENT'] == 1][['SID', 'NAME', 'ISO_TIME', 'WMO_WIND', 'WIND_24H_AHEAD', 'WIND_CHANGE_24H']].head()

## 4. Storm-Independent Train / Val / Test Split

In [ ]:
train_df = df[df['SEASON'] <= 2018]
val_df   = df[(df['SEASON'] >= 2019) & (df['SEASON'] <= 2021)]
test_df  = df[df['SEASON'] >= 2022]

print(f"Train set (1990-2018): {len(train_df)} rows, {train_df['SID'].nunique()} storms")
print(f"Val set   (2019-2021): {len(val_df)} rows, {val_df['SID'].nunique()} storms")
print(f"Test set  (2022-2024): {len(test_df)} rows, {test_df['SID'].nunique()} storms")

# Export splits locally
train_df.to_csv("train_ibtracs.csv", index=False)
val_df.to_csv("val_ibtracs.csv", index=False)
test_df.to_csv("test_ibtracs.csv", index=False)
print("Saved train_ibtracs.csv, val_ibtracs.csv, and test_ibtracs.csv locally!")

## 5. Simple Plots

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(df['WMO_WIND'], bins=20, color='orange', edgecolor='black')
plt.title("Maximum Sustained Wind Speed (knots)")
plt.xlabel("Wind Speed (knots)")
plt.ylabel("Count")
plt.grid(True, alpha=0.3)
plt.show()